# Role attestations

Install Deno and register its TypeScript kernel with `deno jupyter --install`. Open this file
in a Jupyter front end such as JupyterLab or VS Code, select **Deno**, then Run All in a fresh kernel.
The cells import the exact published verifier **2.0.0**.

Verification returns facts; the host still owns trusted inputs, authorization, replay and revocation.
Only safe semantic results are displayed; generated keys, signatures and nonces stay in memory.


## Load the published verifier

Check the package identity and the portable backend Deno selects.


In [ ]:
import { Buffer } from "node:buffer";
import { BACKEND_ID, roleAttestation, verifyGrant } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import { generateKeyPairSync, sign, type KeyObject } from "node:crypto";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
import assert from "node:assert/strict";
assert.equal(verifierPackage.version, "2.0.0");
assert.equal(BACKEND_ID, "portable");
console.log(JSON.stringify({ version: verifierPackage.version, backend: BACKEND_ID }));


{"version":"2.0.0","backend":"portable"}


## Prepare attestor and subject keys

The caller supplies the attestor trust; the subject key is the identity being described.


In [ ]:
const publicKeyBytes = (key: KeyObject): Uint8Array => Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const attestorKeyPair = generateKeyPairSync("ed25519"); // the attestor, for example an organization's root key
const issuerKey = generateKeyPairSync("ed25519"); // the subject
assert.equal(publicKeyBytes(attestorKeyPair.publicKey).length, 32); assert.equal(publicKeyBytes(issuerKey.publicKey).length, 32);
console.log(JSON.stringify({ attestorBytes: publicKeyBytes(attestorKeyPair.publicKey).length, subjectBytes: publicKeyBytes(issuerKey.publicKey).length }));


{"attestorBytes":32,"subjectBytes":32}


## Sign the role attestation

Bind the issuer role to a subject and a half-open validity window.


In [ ]:
// The root key attests that "issuer-2026-10" may act as an issuer from t=1000 until t=2000.
const input = roleAttestation.attestationSigningInput({
  keyId: "root-2026",
  jti: "attestation-0001",
  subjectKeyId: "issuer-2026-10",
  subjectPublicKey: publicKeyBytes(issuerKey.publicKey),
  role: "issuer",
  notBefore: 1_000,
  expiresAt: 2_000,
});
assert.ok(input.ok);
const signingBytes = Buffer.concat([input.value.protectedSegment, Buffer.from("."), input.value.payloadSegment]);
const attestation = roleAttestation.assembleAttestationCompact(input.value, sign(null, signingBytes, attestorKeyPair.privateKey));
assert.ok(attestation.ok);


console.log(JSON.stringify({ ok: attestation.ok }));


{"ok":true}


## Verify the role

A verified role attestation remains diagnostic and does not establish host trust.


In [ ]:
// The verifier supplies the attestor key it trusts (with its validity window), the subject it
// expects, and the time.
const expected = {
  attestor: { keyId: "root-2026", publicKey: publicKeyBytes(attestorKeyPair.publicKey), validFrom: 0, validBefore: null },
  subjectKeyId: "issuer-2026-10",
  subjectPublicKey: publicKeyBytes(issuerKey.publicKey),
  now: 1_500,
};
const facts = roleAttestation.verifyAttestation(attestation.value, expected);
assert.ok(facts.ok);
assert.equal(facts.value.role, "issuer");
assert.equal(facts.value.verification, "signature_and_window");
assert.equal(facts.value.trust, "not_evaluated");


console.log(JSON.stringify({ ok: facts.ok, role: facts.value.role, trust: facts.value.trust }));


{"ok":true,"role":"issuer","trust":"not_evaluated"}


## Refuse expiry

The exact expiry lies outside the half-open window.


In [ ]:
const expired = roleAttestation.verifyAttestation(attestation.value, { ...expected, now: 2_000 });
assert.deepEqual(expired, { ok: false });
console.log(JSON.stringify(expired));


{"ok":false}


## Refuse a different subject

Matching key identifiers cannot substitute another public key.


In [ ]:
const wrongSubject = roleAttestation.verifyAttestation(attestation.value, { ...expected, subjectPublicKey: publicKeyBytes(generateKeyPairSync("ed25519").publicKey) });
assert.deepEqual(wrongSubject, { ok: false });
console.log(JSON.stringify(wrongSubject));


{"ok":false}


## Keep profiles separate

An attestation cannot be used as a grant.


In [ ]:
const notAGrant = verifyGrant(attestation.value, { keyId: "root-2026", publicKey: publicKeyBytes(attestorKeyPair.publicKey) }, { issuer: "root-2026", audience: "x", evaluationTime: 1_500, clockSkew: 0 });
assert.deepEqual(notAGrant, { ok: false });
console.log(JSON.stringify(notAGrant));


{"ok":false}


## Finish

No private material was written to disk. Shut down the kernel to discard the demonstration keys.
Continue with the [getting started guide](../guides/getting-started.md).
